In [5]:
import pandas as pd


import sqlite3

In [8]:
# Connect to the database
DB_PATH = "../../data/private/out/datac88c/fa25/snapshots.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

In [58]:
students_to_include = set()

In [53]:
# only include students with 25 or more backups
students_backup_counts = cursor.execute(
    "SELECT COUNT(*), student_email FROM backup_metadata GROUP BY student_email HAVING COUNT(*) >= 25"
).fetchall()
students_backup_counts

[(33, '005b6cf7'),
 (44, '01b94d64'),
 (77, '01d4e885'),
 (76, '0270c7fc'),
 (93, '027438c4'),
 (68, '0349d6d5'),
 (50, '037c305c'),
 (49, '03a192d5'),
 (150, '05c11b38'),
 (57, '0727b370'),
 (65, '0767c3fd'),
 (53, '07f99937'),
 (86, '08580c5f'),
 (63, '0867310f'),
 (124, '087b234c'),
 (78, '09033ae6'),
 (52, '0992eeb6'),
 (119, '0a060ba3'),
 (112, '0a34fadf'),
 (110, '0ac4827d'),
 (76, '0b397ce1'),
 (58, '0be67f64'),
 (60, '0bf86c20'),
 (75, '0c7da385'),
 (86, '0cbd25bf'),
 (77, '0cd479ba'),
 (150, '0cf1f44e'),
 (77, '0f8cd325'),
 (40, '10e91dd8'),
 (43, '110a0b7a'),
 (51, '113b58ca'),
 (62, '1193e72e'),
 (102, '1249c188'),
 (52, '1258e370'),
 (54, '128d6109'),
 (39, '128fdc9b'),
 (29, '12aab145'),
 (47, '12c5125f'),
 (118, '14b943c8'),
 (103, '15390cdb'),
 (78, '156f0923'),
 (101, '1631e5a3'),
 (68, '183931c4'),
 (42, '1885b853'),
 (71, '190dc9da'),
 (90, '1aedacbd'),
 (33, '1af0e3c9'),
 (112, '1b0917e6'),
 (50, '1b75cdb7'),
 (42, '1baa3af3'),
 (58, '1c238c38'),
 (63, '1cbe76d5'),
 

In [54]:
for count, email in students_backup_counts:
    students_to_include.add(email)

In [55]:
len(students_to_include)

431

In [ ]:
# only include students with lint errors besides F401

# Number of lint errors per student on their final backup, excluding F401
cursor.execute("""
WITH last_backups AS (
    SELECT
        backup_id,
        MAX(created) AS created, -- in SQLite this works
        student_email,
        file_contents_location
    FROM backup_metadata
    GROUP BY student_email
),

backups_with_lint_errors AS (
    SELECT
        lb.backup_id,
        lb.created,
        lb.student_email,
        le.*
    FROM last_backups AS lb
    JOIN lint_errors AS le
    ON REPLACE(lb.file_contents_location, '../../data/private/', '') = REPLACE(le.file_contents_location, '/ants.py', '')
    WHERE le.code != 'F401'
)

SELECT
    student_email,
    COUNT(*)
FROM backups_with_lint_errors
GROUP BY student_email
ORDER BY COUNT(*) DESC
""")
student_to_lint_error_count = cursor.fetchall()
student_to_lint_error_count

[('ad98861e', 18),
 ('bf674fbc', 17),
 ('b30d1d3d', 15),
 ('e52d66d9', 11),
 ('a9286b6c', 11),
 ('9c561a42', 11),
 ('875cb0cd', 11),
 ('83f0d114', 10),
 ('05c11b38', 10),
 ('dc8b5216', 9),
 ('d8328d71', 9),
 ('c91b6b9a', 9),
 ('94ea236e', 8),
 ('9e72fc4e', 7),
 ('547d1da5', 7),
 ('3da25a85', 7),
 ('228021c6', 7),
 ('e1fbf17d', 6),
 ('a97024dc', 6),
 ('a40c28f7', 6),
 ('a2d00ddf', 6),
 ('89c747d1', 6),
 ('443ca939', 6),
 ('419dfa13', 6),
 ('227e6c44', 6),
 ('c6a4f30f', 5),
 ('b646d225', 5),
 ('9737fa20', 5),
 ('6fc50638', 5),
 ('640aa6b9', 5),
 ('5deafc11', 5),
 ('588f72a1', 5),
 ('44434fa4', 5),
 ('0cf1f44e', 5),
 ('f7bd99ce', 4),
 ('e216d5b6', 4),
 ('d1daaba1', 4),
 ('c3d13adf', 4),
 ('9b70ba05', 4),
 ('7d47d37f', 4),
 ('653c46fc', 4),
 ('602dccfe', 4),
 ('4b1352cf', 4),
 ('41f9b5e0', 4),
 ('1ec67d61', 4),
 ('0c7da385', 4),
 ('0659a26e', 4),
 ('f7ffd448', 3),
 ('cdd7659c', 3),
 ('c7042137', 3),
 ('c20830d6', 3),
 ('8635d213', 3),
 ('83f4472b', 3),
 ('7dcfb139', 3),
 ('6ac9b60f', 3),
 

In [59]:
# students_to_include = students_to_include & set([email for email, count in student_to_lint_error_count])
students_to_include = set([email for email, count in student_to_lint_error_count])

len(students_to_include)

187

In [12]:
# print statements


def count_search_string_in_file(path, search_string):
    with open(path, "r") as f:
        contents = f.read()
        return contents.count(search_string)

In [13]:
cursor.execute("""
SELECT backup_id, student_email, created, file_contents_location || '/ants.py' AS file_contents_location
FROM backup_metadata
""")
file_paths = cursor.fetchall()
file_paths[0]

('o7g7AA',
 '9ef88296',
 '2025-11-23T03:47:48',
 '../../data/private/cal/cs88/fa25/ants/9ef88296/o7g7AA/ants.py')

In [ ]:
print_debug_data = {
    "backup_id": [row[0] for row in file_paths if row[3]],
    "student_email": [row[1] for row in file_paths if row[3]],
    "created": [row[2] for row in file_paths if row[3]],
    "num_occurrences_print": [],
}

for path in [row[3] for row in file_paths]:
    if path:
        print_debug_data["num_occurrences_print"].append(
            count_search_string_in_file(path, "print")
        )

In [15]:
print_debug_data_df = pd.DataFrame(print_debug_data)
print_debug_data_df["created"] = pd.to_datetime(print_debug_data_df["created"])
print_debug_data_df.head()

,backup_id,student_email,created,num_occurrences_print
0,o7g7AA,9ef88296,2025-11-23 03:47:48,3
1,mG9G7E,9ef88296,2025-11-23 03:47:37,3
2,lD9D87,9ef88296,2025-11-23 03:47:21,3
3,7OBOD8,9ef88296,2025-11-23 03:46:34,3
4,53z3lq,9ef88296,2025-11-23 03:46:07,3


In [22]:
students_with_high_prints = []
for student_email, num_occurrences_print in zip(
    print_debug_data["student_email"], print_debug_data["num_occurrences_print"]
):
    if num_occurrences_print > 3:
        students_with_high_prints.append([student_email, num_occurrences_print])
students_with_high_prints.sort(key=lambda pair: pair[1], reverse=True)
students_with_high_prints[:10]

[['653c46fc', 9],
 ['653c46fc', 9],
 ['653c46fc', 9],
 ['653c46fc', 9],
 ['653c46fc', 9],
 ['653c46fc', 8],
 ['653c46fc', 8],
 ['653c46fc', 8],
 ['653c46fc', 8],
 ['653c46fc', 8]]

In [60]:
students_to_include = students_to_include & set(
    [email for email, count in students_with_high_prints]
)
len(students_to_include)

18

In [31]:
import random

random.seed(42)

final_list = random.sample(sorted(students_to_include), k=25)
final_list

['d8328d71',
 '252da0ee',
 '087b234c',
 'fa561a78',
 '5deafc11',
 '540e06b1',
 '4c9a38ea',
 '2e65934f',
 'f9e1a41d',
 '209cafa1',
 'e52d66d9',
 'b646d225',
 '1b75cdb7',
 'c91b6b9a',
 '920c0dbd',
 '0a34fadf',
 '0a060ba3',
 '1de91fc6',
 '4b38cff3',
 '50ad8aab',
 'aab5d53b',
 'cdd7659c',
 '09033ae6',
 'bd08e2cf',
 '4572219c']